# Qimono Gate Fidelity: Radar Chart

This notebook reproduces the chart generated by `randomized-benchmarking.py`, but separates the explanation into Markdown cells and the implementation into executable code cells.

**Data source:** `aspects.csv` in this folder. Edit that CSV to change the criteria and qualitative ratings without hard-coding values here.

> **Scientific guardrail:** The ratings are illustrative teaching judgments, not experimental measurements or a universal ranking. A radar chart is a discussion aid, not a rigorous aggregate score.

## 1. Imports and locate the CSV

The notebook uses NumPy for the evenly spaced radar-axis angles, Matplotlib for plotting, and Python's standard-library `csv` module to read the data. It intentionally does not add pandas or SciPy because this chart does not need them.

The path lookup works whether the notebook is launched from the repository root or from inside `gate-fidelity/`.

In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

CSV_CANDIDATES = [Path.cwd() / "aspects.csv", Path.cwd() / "gate-fidelity" / "aspects.csv"]
DATA_PATH = next((path for path in CSV_CANDIDATES if path.is_file()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find aspects.csv. Run this notebook from gate-fidelity/ "
        "or from the repository root."
    )

OUTPUT_PATH = DATA_PATH.with_name("gate-fidelity-radar.png")
print(f"Data file: {DATA_PATH.resolve()}")
print(f"Chart output: {OUTPUT_PATH.resolve()}")

## 2. Load and validate the data

The CSV must contain `criterion`, `randomized_benchmarking`, and `process_tomography` columns. Each score must be numeric and between 1 and 5, and at least three criteria are needed to form a radar chart.

The `notes` and `higher_is_better` columns are useful documentation fields; the current chart displays the scores as supplied and does not transform them.

In [ ]:
METHODS = {
    "Randomized benchmarking": {
        "column": "randomized_benchmarking",
        "color": "#f27fc3",
        "alpha": 0.19,
    },
    "Process tomography": {
        "column": "process_tomography",
        "color": "#5b9cff",
        "alpha": 0.17,
    },
}

criteria = []
ratings = {name: [] for name in METHODS}
required_columns = {"criterion", *(item["column"] for item in METHODS.values())}

with DATA_PATH.open("r", newline="", encoding="utf-8-sig") as csv_file:
    reader = csv.DictReader(csv_file)
    if reader.fieldnames is None or not required_columns.issubset(reader.fieldnames):
        missing = required_columns - set(reader.fieldnames or [])
        raise ValueError(f"CSV is missing required columns: {', '.join(sorted(missing))}")

    for line_number, row in enumerate(reader, start=2):
        criterion = (row.get("criterion") or "").strip()
        if not criterion:
            raise ValueError(f"CSV line {line_number}: criterion cannot be blank.")
        criteria.append(criterion)

        for method, settings in METHODS.items():
            try:
                score = float(row[settings["column"]])
            except (TypeError, ValueError) as exc:
                raise ValueError(
                    f"CSV line {line_number}: {settings['column']} must be numeric."
                ) from exc
            if not 1 <= score <= 5:
                raise ValueError(f"CSV line {line_number}: scores must be between 1 and 5.")
            ratings[method].append(score)

if len(criteria) < 3:
    raise ValueError("A radar chart needs at least three criteria.")

print(f"Loaded {len(criteria)} criteria:")
for criterion in criteria:
    print(f"  • {criterion}")

## 3. Prepare the radar-chart geometry

Each criterion receives an evenly spaced angle around the circle. The first value is repeated at the end of each series so the polygon closes. The score scale is fixed at 1–5 to match `aspects.csv` and the standalone Python script.

In [ ]:
count = len(criteria)
angles = np.linspace(0, 2 * np.pi, count, endpoint=False).tolist()
closed_angles = angles + angles[:1]

fig, ax = plt.subplots(figsize=(9.5, 7.2), subplot_kw={"polar": True})
fig.patch.set_facecolor("#08090d")
ax.set_facecolor("#08090d")
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

ax.set_xticks(angles)
labels = [label.replace(" ", "\n", 1) if len(label) > 17 else label for label in criteria]
ax.set_xticklabels(labels, color="#f1eef6", fontsize=11)
ax.tick_params(axis="x", pad=16)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(["1", "2", "3", "4", "5"], color="#aaa6b4", fontsize=9)
ax.set_rlabel_position(12)
ax.grid(color="#55515e", alpha=0.42, linewidth=0.8)
ax.spines["polar"].set_color("#686371")
ax.spines["polar"].set_alpha(0.5)

## 4. Draw both methods

The visual styling follows the Qimono chart: randomized benchmarking in pink and process tomography in blue. Color and fill are presentation choices; the positions of the vertices come from the CSV scores.

In [ ]:
for name, settings in METHODS.items():
    values = ratings[name]
    closed_values = values + values[:1]
    ax.plot(
        closed_angles,
        closed_values,
        color=settings["color"],
        linewidth=2.6,
        marker="o",
        markersize=6,
        label=name,
    )
    ax.fill(
        closed_angles,
        closed_values,
        color=settings["color"],
        alpha=settings["alpha"],
    )

# Reserve space so the title and top axis label do not collide.
fig.subplots_adjust(top=0.78, bottom=0.27, left=0.08, right=0.92)
fig.suptitle(
    "GATE FIDELITY  /  METHOD TRADE-OFFS",
    color="#f7f3fb",
    fontsize=16,
    fontweight="bold",
    y=0.975,
)
fig.text(
    0.5,
    0.90,
    "Qualitative teaching illustration • scale 1–5 • higher is more favorable",
    ha="center",
    color="#bdb8c7",
    fontsize=9,
)

legend = ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, -0.16),
    ncol=2,
    frameon=False,
    fontsize=10,
    labelcolor="#e8e4ed",
)
for handle in legend.legend_handles:
    handle.set_linewidth(3)

fig.text(
    0.5,
    0.035,
    "Illustrative ratings, not experimental data. Suitability depends on the measurement goal and setup.",
    ha="center",
    color="#ffb98f",
    fontsize=8.5,
)

## 5. Save the chart

The PNG is saved next to the CSV. The cell also displays the chart in the notebook output.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_PATH, dpi=180, facecolor=fig.get_facecolor(), bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Radar chart saved to: {OUTPUT_PATH.resolve()}")

## Interpretation reminder

The course-grounded trade-offs are qualitative: randomized benchmarking is described as more efficient and insensitive to SPAM errors, while process tomography provides a more complete characterization but scales poorly and is sensitive to SPAM errors. The exact numeric scores, including the `Routine monitoring` dimension, are illustrative choices for discussion. Do not interpret polygon area as a rigorous overall score or infer a universal winner or fixed qubit-count threshold.